## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Medical Questions to Answer**

1. **Sepsis Management**: What is the protocol for managing sepsis in a critical care unit?
2. **Appendicitis**: What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
3. **Sudden Patchy Hair Loss**: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?
4. **Brain Tissue Injury**: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?
5. **Leg Fracture During Hiking**: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Uncomment and run only if packages are not yet installed.
# This notebook uses llama-cpp-python with CUDA support for GPU inference.

# GPU build (CUDA 12.x):
# !CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install llama-cpp-python --force-reinstall --no-cache-dir -q

# Core libraries:
# !pip install huggingface_hub pandas tiktoken pymupdf langchain langchain-community chromadb sentence-transformers -q

**Note**: After running the installation cell, restart the notebook kernel and run all cells sequentially from the next cell onward. A warning about package dependencies may appear — it can be safely ignored as long as all imports in the next cell succeed.

In [2]:
# Libraries for processing dataframes and text
import json
import os
import textwrap
import warnings

# Prevent sentence-transformers from attempting to load TensorFlow/tf_keras
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import pandas as pd
import tiktoken
from IPython.display import Markdown, display

# Libraries for loading data, chunking, embeddings, and vector databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

# Libraries for downloading and loading the LLM directly from Hugging Face Hub
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 180)

**Note**: All imports above are from standard open-source libraries. `huggingface_hub` downloads the GGUF model directly from the Hugging Face Hub; `llama_cpp` runs it locally with GPU/CPU inference. No Ollama or remote API server is required.

In [3]:
# Set Hugging Face cache paths so models are stored in a known location
os.environ["HF_HOME"] = r"F:\AI_Cache\huggingface"
os.environ["HUGGINGFACE_HUB_CACHE"] = r"F:\AI_Cache\huggingface\hub"
os.environ["TRANSFORMERS_CACHE"] = r"F:\AI_Cache\huggingface\transformers"
os.environ["HF_DATASETS_CACHE"] = r"F:\AI_Cache\huggingface\datasets"
os.environ["TORCH_HOME"] = r"F:\AI_Cache\torch"

PROJECT_DIR = r"D:\pythonds\TEXAS-DSBA\Medical Assistant - NLP with Generative AI"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
CHROMA_DB_DIR = os.path.join(PROJECT_DIR, "chroma_db")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHROMA_DB_DIR, exist_ok=True)

print("Environment configured.")
print(f"Data directory  : {DATA_DIR}")
print(f"ChromaDB directory: {CHROMA_DB_DIR}")

Environment configured.
Data directory  : D:\pythonds\TEXAS-DSBA\Medical Assistant - NLP with Generative AI\data
ChromaDB directory: D:\pythonds\TEXAS-DSBA\Medical Assistant - NLP with Generative AI\chroma_db


## Question Answering using LLM

### Downloading and Loading the Model

The model is downloaded directly from the **Hugging Face Hub** using `hf_hub_download` and loaded locally via `llama_cpp`. No external server (Ollama, OpenAI API, etc.) is required.

**Model**: Mistral-7B-Instruct-v0.2 (Q6_K GGUF quantization)  
**Source**: `TheBloke/Mistral-7B-Instruct-v0.2-GGUF` on Hugging Face Hub  
**Why Mistral-7B**: Strong instruction-following capability; well-suited for structured medical Q&A; Q6_K quantization balances quality and VRAM usage (~5.5 GB).

In [4]:
MODEL_REPO_ID = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
MODEL_FILE   = "mistral-7b-instruct-v0.2.Q6_K.gguf"

# Download from Hugging Face Hub (cached after first download — no re-download on subsequent runs)
model_path = hf_hub_download(
    repo_id=MODEL_REPO_ID,
    filename=MODEL_FILE,
    cache_dir=os.environ.get("HUGGINGFACE_HUB_CACHE", None)
)
print(f"Model cached at: {model_path}")

# Load the model with GPU offloading
# n_gpu_layers=32 offloads all layers to RTX 3070 (8 GB VRAM).
# Reduce to 16 or 0 if VRAM is insufficient.
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_gpu_layers=32,
    n_batch=512,
    verbose=False,
)
print("Model loaded successfully.")

Model cached at: F:\AI_Cache\huggingface\hub\models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF\snapshots\3a6fbf4a41a1d52e415a4958cde6856d34b2db93\mistral-7b-instruct-v0.2.Q6_K.gguf


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Model loaded successfully.


**Observation — Model Loading**: The model is fetched from the official Hugging Face Hub (`TheBloke/Mistral-7B-Instruct-v0.2-GGUF`) and cached locally. Once cached, the notebook runs fully offline. The Q6_K quantisation reduces the weight from ~14 GB (fp16) to ~5.5 GB with minimal degradation in output quality for instruction-following tasks.

In [5]:
# Five medical questions from the project brief
questions = {
    "Q1": "What is the protocol for managing sepsis in a critical care unit?",
    "Q2": (
        "What are the common symptoms of appendicitis, and can it be cured via medicine? "
        "If not, what surgical procedure should be followed to treat it?"
    ),
    "Q3": (
        "What are the effective treatments or solutions for addressing sudden patchy hair loss, "
        "commonly seen as localized bald spots on the scalp, "
        "and what could be the possible causes behind it?"
    ),
    "Q4": (
        "What treatments are recommended for a person who has sustained a physical injury "
        "to brain tissue, resulting in temporary or permanent impairment of brain function?"
    ),
    "Q5": (
        "What are the necessary precautions and treatment steps for a person who has "
        "fractured their leg during a hiking trip, and what should be considered for "
        "their care and recovery?"
    ),
}

for key, q in questions.items():
    print(f"{key}: {q}")

Q1: What is the protocol for managing sepsis in a critical care unit?
Q2: What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
Q3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?
Q4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?
Q5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?


#### Response Function

In [6]:
def _format_mistral_prompt(system_message: str, user_message: str) -> str:
    """Wrap messages in Mistral instruction-tuning chat format."""
    return f"<s>[INST] {system_message}\n\n{user_message} [/INST]"


def llm_chat(
    system_message: str,
    user_message: str,
    max_tokens: int = 350,
    temperature: float = 0.1,
    top_p: float = 0.95,
    top_k: int = 50,
) -> str:
    """Call the locally loaded Hugging Face GGUF model and return the response text."""
    prompt = _format_mistral_prompt(system_message, user_message)
    output = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        stop=["[INST]", "</s>"],
        echo=False,
    )
    return output["choices"][0]["text"].strip()


def response(
    query: str,
    max_tokens: int = 350,
    temperature: float = 0.1,
    top_p: float = 0.95,
    top_k: int = 50,
) -> str:
    """Generate a plain base-LLM response with no external retrieved context."""
    return llm_chat(
        system_message=(
            "You are a careful medical information assistant. "
            "Give accurate, concise, safety-aware answers and always recommend "
            "licensed clinical judgment for all medical decisions."
        ),
        user_message=query,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
    )


print("Functions defined: _format_mistral_prompt | llm_chat | response")

Functions defined: _format_mistral_prompt | llm_chat | response


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [7]:
base_llm_answers = globals().get("base_llm_answers", {})
base_llm_answers["Q1"] = response(questions["Q1"], max_tokens=420, temperature=0.1)
print(base_llm_answers["Q1"])

Sepsis is a life-threatening condition caused by the body's response to an infection. In a critical care unit, managing sepsis involves a multidisciplinary approach and prompt recognition and intervention. Here are the general steps for managing sepsis in a critical care unit:

1. Early recognition: Recognize sepsis early by assessing for signs and symptoms such as fever, chills, tachycardia, tachypnea, altered mental status, and lactic acidosis. Use the Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score or the Quick Sequential [Sepsis-related] Organ Failure Assessment (qSOFA) score to help identify patients at risk for sepsis.
2. Immediate fluid resuscitation: Administer intravenous fluids to maintain adequate tissue perfusion and blood pressure. The goal is to achieve a mean arterial pressure (MAP) of 65 mmHg or higher and a central venous pressure (CVP) of 8-12 mmHg.
3. Antibiotics: Administer broad-spectrum antibiotics as soon as possible based on the patient's clini

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [8]:
base_llm_answers = globals().get("base_llm_answers", {})
base_llm_answers["Q2"] = response(questions["Q2"], max_tokens=420, temperature=0.1)
print(base_llm_answers["Q2"])

Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right abdomen. Common symptoms of appendicitis include:

1. Abdominal pain, usually starting around the navel and then shifting to the lower right abdomen.
2. Loss of appetite.
3. Nausea and vomiting.
4. Fever.
5. Diarrhea or constipation.
6. Abdominal swelling and tenderness.
7. Inability to pass gas or have a bowel movement.

Appendicitis cannot be cured with medicine alone. If left untreated, the appendix may rupture, leading to peritonitis, a serious inflammation of the abdominal cavity. The standard treatment for appendicitis is surgical removal of the appendix, known as an appendectomy. This procedure can be performed as an open appendectomy or laparoscopically, depending on the severity of the condition. It is essential to consult a healthcare professional for proper diagnosis and treatment.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [9]:
base_llm_answers = globals().get("base_llm_answers", {})
base_llm_answers["Q3"] = response(questions["Q3"], max_tokens=420, temperature=0.1)
print(base_llm_answers["Q3"])

I cannot provide specific treatments or solutions without a proper diagnosis from a licensed healthcare professional. Sudden patchy hair loss, also known as alopecia areata, can have various causes, including autoimmune disorders, stress, vitamin deficiencies, or certain medications. It's essential to consult a healthcare provider or a dermatologist for an accurate diagnosis and appropriate treatment options.

Some common treatments for alopecia areata include:

1. Topical treatments: Minoxidil (Rogaine) or corticosteroid creams or ointments can be applied directly to the affected area to stimulate hair growth.
2. Injections: Corticosteroids can be injected into the bald spots to reduce inflammation and promote hair regrowth.
3. Systemic treatments: Oral corticosteroids, immunosuppressive drugs, or biologic agents may be prescribed for severe cases or when other treatments are ineffective.
4. Alternative therapies: Acupuncture, wigs, or hairpieces can help manage the appearance of hair

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [10]:
base_llm_answers = globals().get("base_llm_answers", {})
base_llm_answers["Q4"] = response(questions["Q4"], max_tokens=420, temperature=0.1)
print(base_llm_answers["Q4"])

I cannot provide specific treatment recommendations without a thorough evaluation by a licensed healthcare professional. However, I can provide some general information about common treatments for brain injuries.

1. Medical stabilization: The first priority is to ensure the person's airway is clear, breathing is stable, and circulation is adequate. This may involve emergency procedures such as intubation, surgery, or administration of medications.

2. Rehabilitation: Rehabilitation is an essential part of treatment for brain injuries. Rehabilitation may include physical therapy, occupational therapy, speech therapy, and cognitive rehabilitation to help the person regain lost skills and improve function.

3. Medications: Depending on the specific symptoms, medications may be prescribed to manage conditions such as seizures, pain, or depression.

4. Surgery: In some cases, surgery may be necessary to remove blood clots, repair skull fractures, or relieve pressure on the brain.

5. Assis

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [11]:
base_llm_answers = globals().get("base_llm_answers", {})
base_llm_answers["Q5"] = response(questions["Q5"], max_tokens=420, temperature=0.1)
print(base_llm_answers["Q5"])

I'm an AI language model and not a doctor, but I can provide you with general information on this topic based on available resources. If someone has fractured their leg during a hiking trip, here are some necessary precautions and treatment steps:

1. Assess the severity of the injury: If the fracture is open (compound), if there is significant swelling, or if the person is experiencing severe pain, they should seek emergency medical help immediately.

2. Keep the leg stable: Apply a splint or use a makeshift sling to keep the leg stable and prevent any further movement. This will help reduce pain and prevent the fracture from worsening.

3. Control swelling: Elevate the injured leg above heart level to help reduce swelling. Apply ice packs for 15-20 minutes at a time, several times a day, to help reduce inflammation.

4. Pain management: Over-the-counter pain relievers, such as acetaminophen or ibuprofen, can help manage pain. However, consult a healthcare professional before taking a

**Observation — Base LLM (No Context)**: The model generates plausible medical information based purely on its training data. However, responses may:

- Lack specificity from the Merck Manual.
- Occasionally introduce unsupported or out-of-date claims.
- Miss protocol-level detail for critical questions such as sepsis management.

This highlights why grounding the model in a trusted reference document (RAG) is essential for clinical decision support.

## Question Answering using LLM with Prompt Engineering

Five distinct prompt profiles are defined — each combines a different system message with different generation parameters (temperature, top_p, top_k, max_tokens). Each of the five medical questions is answered with a different profile to demonstrate the range of prompt engineering effects.

In [12]:
# Five prompt / parameter combinations
prompt_profiles = [
    {
        "name": "concise_clinical_protocol",
        "system": (
            "You are a careful clinical decision-support assistant. "
            "Give a concise protocol-style answer and include emergency red flags."
        ),
        "temperature": 0.0, "top_p": 0.90, "top_k": 30, "max_tokens": 420,
    },
    {
        "name": "differential_and_treatment",
        "system": (
            "You are a medical assistant helping clinicians. "
            "Structure the answer as: symptoms, likely diagnosis, treatment, "
            "and when to escalate care."
        ),
        "temperature": 0.2, "top_p": 0.95, "top_k": 40, "max_tokens": 460,
    },
    {
        "name": "patient_safe_language",
        "system": (
            "You are a healthcare information assistant. "
            "Use patient-friendly language, avoid definitive diagnosis, "
            "and recommend professional medical evaluation."
        ),
        "temperature": 0.3, "top_p": 0.95, "top_k": 50, "max_tokens": 460,
    },
    {
        "name": "trauma_first_aid",
        "system": (
            "You are an emergency medicine support assistant. "
            "Prioritize stabilization, immediate precautions, and referral to "
            "urgent or emergency care."
        ),
        "temperature": 0.1, "top_p": 0.85, "top_k": 30, "max_tokens": 460,
    },
    {
        "name": "evidence_checklist",
        "system": (
            "You are a clinical knowledge assistant. "
            "Answer with a checklist of key findings, treatment steps, "
            "contraindications, and follow-up considerations."
        ),
        "temperature": 0.0, "top_p": 0.92, "top_k": 20, "max_tokens": 500,
    },
]


def prompt_engineered_response(question: str, profile: dict) -> str:
    """Apply a specific prompt / parameter profile to generate an engineered response."""
    return llm_chat(
        system_message=profile["system"],
        user_message=question,
        max_tokens=profile["max_tokens"],
        temperature=profile["temperature"],
        top_p=profile["top_p"],
        top_k=profile["top_k"],
    )


print(f"Defined {len(prompt_profiles)} prompt engineering profiles.")

Defined 5 prompt engineering profiles.


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [13]:
prompt_engineered_answers = globals().get("prompt_engineered_answers", {})
prompt_engineered_answers["Q1"] = {
    "profile": prompt_profiles[0]["name"],
    "answer": prompt_engineered_response(questions["Q1"], prompt_profiles[0])
}
print(f"Profile : {prompt_engineered_answers['Q1']['profile']}")
print(prompt_engineered_answers["Q1"]["answer"])

Profile : concise_clinical_protocol
Title: Sepsis Management Protocol in Critical Care Unit

1. **Recognition:**
   - Suspect sepsis in any patient with suspected or confirmed infection and organ dysfunction.
   - Use the Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score to identify organ dysfunction.
   - Consider septic shock if the patient has sepsis and requires vasopressors to maintain mean arterial pressure (MAP) ≥65 mmHg and/or has serum lactate level >2 mmol/L despite adequate volume resuscitation.

2. **Immediate Actions:**
   - High-flow oxygen via a non-rebreather mask or endotracheal tube.
   - Initiate intravenous (IV) access and administer 30 mL/kg of crystalloid solution within the first hour.
   - Administer broad-spectrum antibiotics based on suspected infection site and microbiology.
   - Start IV fluid resuscitation to maintain MAP ≥65 mmHg and urine output ≥0.5 mL/kg/h.
   - Monitor and treat for hypothermia (core temperature <36°C).

3. **Fluid Resu

**Observation — Q1**: Profile **concise_clinical_protocol** (temp=0.0, top_k=30): deterministic, terse protocol output. Ideal for ICU dashboards where clinicians need quick reference bullets.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [14]:
prompt_engineered_answers = globals().get("prompt_engineered_answers", {})
prompt_engineered_answers["Q2"] = {
    "profile": prompt_profiles[1]["name"],
    "answer": prompt_engineered_response(questions["Q2"], prompt_profiles[1])
}
print(f"Profile : {prompt_engineered_answers['Q2']['profile']}")
print(prompt_engineered_answers["Q2"]["answer"])

Profile : differential_and_treatment
Symptoms of Appendicitis:
1. Abdominal pain, usually starting around the navel and then shifting to the lower right side.
2. Loss of appetite.
3. Nausea and vomiting.
4. Fever (often low-grade at first, but can rise as high as 101°F or 38.3°F).
5. Abdominal swelling and rigidity.
6. Inability to pass gas or have a bowel movement.
7. Diarrhea or constipation.
8. Feeling ill or having a general sense of malaise.

Appendicitis cannot be cured via medicine alone. The appendix is a small, tube-like structure at the end of the large intestine. When it becomes inflamed or infected, it must be removed to prevent rupture and potential complications such as peritonitis. The standard treatment for appendicitis is an appendectomy, a surgical procedure to remove the appendix.

When to escalate care:
1. Severe abdominal pain that is not relieved by lying down or taking pain medication.
2. Fever above 101°F (38.3°C).
3. Vomiting repeatedly and unable to keep fluid

**Observation — Q2**: Profile **differential_and_treatment** (temp=0.2, top_k=40): lightly exploratory. Structured into symptoms → diagnosis → treatment → escalation, matching clinical SOAP-note thinking.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [15]:
prompt_engineered_answers = globals().get("prompt_engineered_answers", {})
prompt_engineered_answers["Q3"] = {
    "profile": prompt_profiles[2]["name"],
    "answer": prompt_engineered_response(questions["Q3"], prompt_profiles[2])
}
print(f"Profile : {prompt_engineered_answers['Q3']['profile']}")
print(prompt_engineered_answers["Q3"]["answer"])

Profile : patient_safe_language
I'm glad you've asked about sudden patchy hair loss, also known as alopecia areata. This condition is characterized by round, smooth patches of baldness on the scalp or other areas of the body. The cause of alopecia areata is not fully understood, but it's believed to be an autoimmune disease, meaning the body's immune system attacks the hair follicles.

There are several treatments that may help with alopecia areata, but it's important to note that each person's experience with the condition can be different. Here are some options that have shown promise for some people:

1. Topical treatments: Corticosteroids, such as minoxidil, can be applied directly to the affected area to help stimulate hair growth.
2. Injections: Corticosteroid injections directly into the bald spots can be effective for some people, but they may cause side effects and require multiple treatments.
3. Systemic treatments: Oral corticosteroids, immunosuppressive drugs, or biologic a

**Observation — Q3**: Profile **patient_safe_language** (temp=0.3, top_k=50): more creative phrasing, accessible vocabulary. Higher temperature allows varied, conversational output suitable for patient-facing portals.

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [16]:
prompt_engineered_answers = globals().get("prompt_engineered_answers", {})
prompt_engineered_answers["Q4"] = {
    "profile": prompt_profiles[3]["name"],
    "answer": prompt_engineered_response(questions["Q4"], prompt_profiles[3])
}
print(f"Profile : {prompt_engineered_answers['Q4']['profile']}")
print(prompt_engineered_answers["Q4"]["answer"])

Profile : trauma_first_aid
A person with a brain injury should receive immediate medical attention. Here are the recommended treatments and precautions:

1. Airway and Breathing: Ensure the person's airway is open and they are breathing. If not breathing, initiate CPR (Cardiopulmonary Resuscitation) if trained to do so.

2. Circulation: Control any external bleeding and monitor vital signs.

3. Disability assessment: Assess the level of consciousness using the AVPU (Alert, Voice, Pain, Unresponsive) scale.

4. Immobilize the head and neck: Use a cervical collar and a rigid backboard if there is suspicion of a spinal injury.

5. Oxygen: Provide supplemental oxygen if the person is not breathing adequately.

6. Glucose: Give a sugar solution (glucose) if the person is unconscious or unresponsive.

7. Medications: Administer medications as prescribed by a healthcare professional to manage symptoms such as increased intracranial pressure, seizures, or other complications.

8. Monitor vital

**Observation — Q4**: Profile **trauma_first_aid** (temp=0.1, top_p=0.85): conservative and action-oriented. Lower top_p restricts token sampling to high-probability continuations, reducing risk of hallucination in emergencies.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [17]:
prompt_engineered_answers = globals().get("prompt_engineered_answers", {})
prompt_engineered_answers["Q5"] = {
    "profile": prompt_profiles[4]["name"],
    "answer": prompt_engineered_response(questions["Q5"], prompt_profiles[4])
}
print(f"Profile : {prompt_engineered_answers['Q5']['profile']}")
print(prompt_engineered_answers["Q5"]["answer"])

Profile : evidence_checklist
Title: Leg Fracture: Necessary Precautions, Treatment Steps, Contraindications, and Follow-Up Considerations

I. Key Findings:
1. A leg fracture is a common injury during hiking, often caused by falls, tripping over rocks, or twisting motions.
2. Symptoms may include pain, swelling, deformity, difficulty walking, and inability to bear weight on the affected leg.
3. Open fractures (compound fractures) may involve the skin being pierced by the bone, increasing the risk of infection.

II. Necessary Precautions:
1. Maintain a stable environment: Ensure the person is lying down with the injured leg elevated and supported.
2. Immobilize the leg: Use a splint, sling, or brace to prevent further movement and reduce pain.
3. Apply ice: Apply ice packs to the affected area for 15-20 minutes at a time, several times a day, to help reduce swelling and pain.
4. Avoid weight-bearing activities: Encourage the person to avoid putting weight on the affected leg to prevent f

**Observation — Q5**: Profile **evidence_checklist** (temp=0.0, top_k=20): maximally deterministic. Checklist format is ideal for structured care plans, discharge instructions, and multi-step recovery management.

**Overall Observation — Prompt Engineering**: Prompt engineering substantially shapes the tone, structure, and safety framing of LLM responses without changing the underlying model weights. However, prompt-engineered answers are still limited to the model's parametric (training-time) knowledge. For medical contexts, this means responses may miss specific protocol details from specialized references such as the Merck Manual. RAG (next section) addresses this limitation by grounding responses in retrieved source text.

## Data Preparation for RAG

### Loading the Data

In [18]:
# The Merck Manual PDF must be placed in the data directory.
# Adjust the filename below if your copy has a different name.
manual_pdf_path = os.path.join(DATA_DIR, "medical_diagnosis_manual.pdf")

pdf_loader = PyMuPDFLoader(manual_pdf_path)
manual = pdf_loader.load()

print(f"PDF loaded from : {manual_pdf_path}")

PDF loaded from : D:\pythonds\TEXAS-DSBA\Medical Assistant - NLP with Generative AI\data\medical_diagnosis_manual.pdf


### Data Overview

#### Checking the first 5 pages

In [19]:
for i, page in enumerate(manual[:5], start=1):
    print(f"\n--- Page {i} ---")
    print(page.page_content[:1200])


--- Page 1 ---
akhtar.naheed@gmail.com
HF24TEQP91
This file is meant for personal use by akhtar.naheed@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

--- Page 2 ---
akhtar.naheed@gmail.com
HF24TEQP91
This file is meant for personal use by akhtar.naheed@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

--- Page 3 ---
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .......................................................................................................................

#### Checking the number of pages

In [20]:
print(f"Number of pages in the manual: {len(manual):,}")

Number of pages in the manual: 4,114


**Observation — Data Overview**: The Merck Manual spans thousands of pages covering 23 clinical sections. Page content is largely unstructured running text with embedded headers, tables, and drug names. This confirms that chunking with overlap is necessary to preserve context across section boundaries.

### Data Chunking

In [21]:
# Split pages into overlapping chunks so each retrieved passage retains enough clinical context.
# chunk_size=700 tokens balances retrieval precision with enough content per chunk.
# chunk_overlap=120 tokens preserves treatment steps that span paragraph boundaries.
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=700,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""]
)

document_chunks = pdf_loader.load_and_split(text_splitter)
print(f"Created {len(document_chunks):,} chunks")

print("\n--- Chunk 0 preview ---")
print(document_chunks[0].page_content[:800])

Created 7,203 chunks

--- Chunk 0 preview ---
akhtar.naheed@gmail.com
HF24TEQP91
This file is meant for personal use by akhtar.naheed@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.


**Observation — Chunking**: The `RecursiveCharacterTextSplitter` respects natural text boundaries (paragraphs, sentences) before resorting to word splits. The 120-token overlap ensures that a clinical step described at the end of one chunk is also present at the start of the next, preventing retrieval misses for multi-step protocols.

### Embedding

In [22]:
# Ensure TensorFlow is not loaded by sentence-transformers
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

# Load the sentence-transformers embedding model
# all-MiniLM-L6-v2: 384-dim dense vectors, fast, accurate for semantic similarity
embedding_model = SentenceTransformerEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

print("Embedding model : sentence-transformers/all-MiniLM-L6-v2")
print(f"Dimension       : {len(embedding_1)}")
print(f"Consistent dims : {len(embedding_1) == len(embedding_2)}")

Embedding model : sentence-transformers/all-MiniLM-L6-v2
Dimension       : 384
Consistent dims : True


**Observation — Embedding**: `all-MiniLM-L6-v2` produces 384-dimensional dense vectors that capture semantic similarity rather than keyword overlap. Two chunks about the same clinical concept will have high cosine similarity even if they use different terminology — essential for medical text where synonyms (e.g. 'myocardial infarction' vs 'heart attack') must match.

### Vector Database

In [23]:
# Build and persist the Chroma vector database
out_dir = CHROMA_DB_DIR
os.makedirs(out_dir, exist_ok=True)

vectorstore = Chroma.from_documents(
    documents=document_chunks,
    embedding=embedding_model,
    persist_directory=out_dir
)

# persist() is required in older Chroma versions; no-op in newer ones
if hasattr(vectorstore, "persist"):
    vectorstore.persist()

print(f"Vector database created at: {out_dir}")

Vector database created at: D:\pythonds\TEXAS-DSBA\Medical Assistant - NLP with Generative AI\chroma_db


**Observation — Vector Database**: Chroma stores both the chunk text and its embedding vector, enabling fast approximate-nearest-neighbour (ANN) search at query time. Persisting to disk means the vector index survives kernel restarts — avoiding the need to re-embed the entire 4,000+ page manual on every run.

### Retriever

In [24]:
# Configure the default retriever: cosine-similarity search returning k=4 chunks
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

# Smoke-test: retrieve chunks for Q1 and inspect source pages
sample_query = questions["Q1"]
rel_docs = vectorstore.similarity_search(sample_query, k=4)

for i, doc in enumerate(rel_docs, start=1):
    page = doc.metadata.get("page", "unknown")
    print(f"\n--- Retrieved chunk {i}; page {page} ---")
    print(doc.page_content[:700])


--- Retrieved chunk 1; page 2400 ---
16 - Critical Care Medicine
Chapter 222. Approach to the Critically Ill Patient
Introduction
Critical care medicine specializes in caring for the most seriously ill patients. These patients are best
treated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special
populations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high
nurse:patient ratio to provide the necessary high intensity of service, including treatment and monitoring
of physiologic parameters.
Supportive care for the ICU patient includes provision of adequate nutrition (see p. 21) and prevention of
infection, stress ulcers and gastritis (see p. 131), and p

--- Retrieved chunk 2; page 2400 ---
16 - Critical Care Medicine
Chapter 222. Approach to the Critically Ill Patient
Introduction
Critical care medicine specializes in caring for the most seriously ill patients. These patients are best
treated in an ICU staffe

**Observation — Retriever**: `search_type='similarity'` performs cosine-similarity matching. Setting k=4 retrieves enough evidence to answer multi-part questions while keeping the prompt concise; with n_ctx=4096 the full RAG prompt comfortably fits. In the fine-tuning section we also test MMR (Maximum Marginal Relevance), which penalises redundant chunks and improves coverage for questions with multiple subtopics.

### System and User Prompt Template

In [25]:
qna_system_message = """
You are a medical decision-support assistant using a trusted medical manual as context.
Answer only from the supplied context when possible. If the context is incomplete, say what is
missing instead of inventing details.
Use clear clinical language, include immediate safety precautions when relevant, and remind the
user that the output supports but does not replace licensed clinical judgment.
""".strip()

qna_user_message_template = """
Context from the medical manual:
{context}

Question:
{question}

Provide a grounded answer with these sections when applicable:
- Direct answer
- Symptoms or causes
- Treatment / management steps
- Urgent escalation criteria
- Source pages used
""".strip()

print("System message and user prompt template defined.")

System message and user prompt template defined.


### Response Function

In [26]:
def _format_docs_for_context(docs: list) -> str:
    """Concatenate retrieved chunks with their source metadata as headers."""
    newline = chr(10)
    formatted_chunks = []
    for i, doc in enumerate(docs, start=1):
        page = doc.metadata.get("page")
        source = doc.metadata.get("source", "medical manual")
        page_label = page + 1 if isinstance(page, int) else page
        header = f"[Source {i}: {source}, page {page_label}]"
        formatted_chunks.append(header + newline + doc.page_content)
    return (newline * 2).join(formatted_chunks)


def retrieve_documents(
    user_input: str,
    k: int = 4,
    search_type: str = "similarity",
    fetch_k: int = 12,
) -> list:
    """Retrieve relevant chunks using similarity or MMR search."""
    if search_type == "mmr":
        return vectorstore.max_marginal_relevance_search(
            user_input, k=k, fetch_k=fetch_k
        )
    return vectorstore.similarity_search(user_input, k=k)


def generate_rag_response(
    user_input: str,
    k: int = 4,
    search_type: str = "similarity",
    fetch_k: int = 12,
    max_tokens: int = 550,
    temperature: float = 0.0,
    top_p: float = 0.90,
    top_k: int = 30,
    return_context: bool = False,
) -> str:
    """Retrieve relevant passages from the Merck Manual and generate a grounded answer."""
    relevant_document_chunks = retrieve_documents(
        user_input=user_input,
        k=k,
        search_type=search_type,
        fetch_k=fetch_k,
    )
    context_for_query = _format_docs_for_context(relevant_document_chunks)

    user_message = qna_user_message_template.format(
        context=context_for_query,
        question=user_input,
    )

    try:
        answer = llm_chat(
            system_message=qna_system_message,
            user_message=user_message,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
        )
    except Exception as e:
        answer = f"Sorry, I encountered the following error:\n{e}"

    if return_context:
        return answer, context_for_query, relevant_document_chunks
    return answer


print("RAG functions defined: retrieve_documents | _format_docs_for_context | generate_rag_response")

RAG functions defined: retrieve_documents | _format_docs_for_context | generate_rag_response


## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [27]:
rag_answers = globals().get("rag_answers", {})
rag_answers["Q1"] = generate_rag_response(
    questions["Q1"], k=4, search_type="similarity", max_tokens=650
)
print(rag_answers["Q1"])

Direct answer:
The protocol for managing sepsis in a critical care unit involves the following steps:

Symptoms or causes:
Sepsis is a life-threatening condition caused by a severe infection. The symptoms include shaking chills, persistent fever, altered sensorium, hypotension, and GI symptoms such as abdominal pain, nausea, vomiting, and diarrhea. Septic shock develops in 25 to 40% of patients with significant bacteremia.

Treatment / management steps:
1. Obtain cultures of blood and any other appropriate specimens.
2. Administer empiric antibiotics after appropriate cultures are obtained.
3. Adjust antibiotics according to the results of culture and susceptibility testing.
4. Surgically drain any abscesses.
5. Remove any internal devices that are the suspected source of bacteria.

Urgent escalation criteria:
If sepsis or septic shock is suspected, immediate action is required. The patient should be transferred to the ICU for close monitoring and intensive care. The healthcare team sh

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [28]:
rag_answers = globals().get("rag_answers", {})
rag_answers["Q2"] = generate_rag_response(
    questions["Q2"], k=4, search_type="similarity", max_tokens=650
)
print(rag_answers["Q2"])

The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which is then followed by pain shifting to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point. Additional signs include pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right hip joint (psoas sign), or pain caused by passive internal rotation of the flexed thigh (obturator sign). A low-grade fever (rectal temperature 37.7 to 38.3° C [100 to 101° F]) is common. However, these classic findings appear in less than 50% of patients, and many variations of symptoms and signs occur.

Appendicitis cannot be cured via medicine alone. The treatment for appendicitis is open or laparoscopic appendectomy. Because treatment delay increases mortality, a negative appendecto

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [29]:
rag_answers = globals().get("rag_answers", {})
rag_answers["Q3"] = generate_rag_response(
    questions["Q3"], k=4, search_type="similarity", max_tokens=650
)
print(rag_answers["Q3"])

Direct answer:
Sudden patchy hair loss, also known as alopecia areata, is a condition characterized by the sudden appearance of bald spots on the scalp or other hair-bearing areas of the body. This condition is usually not associated with any obvious skin or systemic disorders. Alopecia areata is believed to be an autoimmune disorder that affects genetically susceptible individuals and may be triggered by unclear environmental factors.

Symptoms or causes:
The symptoms of alopecia areata include sudden onset of round or oval bald patches on the scalp or other hair-bearing areas. The hair loss may affect the scalp, beard, or any other part of the body. In severe cases, the hair loss may involve the entire scalp (alopecia universalis) or the entire body (alopecia totalis).

Treatment / management steps:
The treatment for alopecia areata depends on the severity and extent of the hair loss. In mild cases, no treatment may be necessary as the hair loss is often temporary and may spontaneous

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [30]:
rag_answers = globals().get("rag_answers", {})
rag_answers["Q4"] = generate_rag_response(
    questions["Q4"], k=4, search_type="similarity", max_tokens=650
)
print(rag_answers["Q4"])

Direct answer:
The treatment for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, depends on the nature and extent of the damage. Recovery from brain injury depends on the plasticity of the remaining cerebrum, which varies from person to person and is affected by age and general health. Plasticity is most prominent in the developing brain. While capacity for recovery from brain injury is considerable after the first decade of life, severe damage more often results in permanent deficits. Gross reorganization of brain function after injury in adults is uncommon, but plasticity remains operative in certain specific areas of the brain throughout life.

Symptoms or causes:
Brain injury can result from various causes, including trauma, toxicmetabolic disorders, vasculopathy, major trauma, or disseminated cancer. The symptoms can range from mild to severe and may include impairment of cognitive function, speech, me

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [31]:
rag_answers = globals().get("rag_answers", {})
rag_answers["Q5"] = generate_rag_response(
    questions["Q5"], k=4, search_type="similarity", max_tokens=650
)
print(rag_answers["Q5"])

Direct answer:
For a person who has fractured their leg during a hiking trip, the following steps should be taken:
1. Assess the severity of the injury and provide initial first aid, which includes immobilizing the affected limb using a non-rigid or non-circumferential splint to prevent further injury and decrease pain.
2. Apply RICE (Rest, Ice, Compression, and Elevation) principles to minimize swelling and pain. Ice should be applied intermittently for 15 to 20 minutes, as often as possible during the first 24 to 48 hours. Compression can be achieved using a splint, elastic bandage, or a Jones compression dressing. The injured limb should be elevated above the heart for the first 2 days to minimize swelling.
3. Pain management: Pain is typically treated with opioids, but the source provides no specific dosage or administration instructions. Consult a healthcare professional for appropriate pain management.
4. Seek medical attention as soon as possible for definitive treatment, which 

**Observation — RAG vs Base LLM**: RAG answers are noticeably more specific and grounded. Responses cite source page ranges from the Merck Manual, include step-by-step protocol details, and flag escalation criteria that the base LLM omitted. The improvement is most pronounced for complex multi-step questions (sepsis, brain injury) where the manual's structured protocol provides content the model's parametric memory cannot match reliably.

### Fine-tuning

Five RAG configurations are tested by varying `k`, `search_type`, `temperature`, `top_p`, `top_k`, and `max_tokens`. Results are displayed in a summary DataFrame so the best-performing configuration per question type can be identified.

In [ ]:
# Five RAG tuning profiles — similarity vs MMR, conservative vs exploratory generation
rag_tuning_profiles = [
    {
        "name": "focused_similarity",
        "k": 3, "search_type": "similarity", "fetch_k": 10,
        "temperature": 0.0, "top_p": 0.85, "top_k": 20, "max_tokens": 500,
    },
    {
        "name": "broader_similarity",
        "k": 5, "search_type": "similarity", "fetch_k": 12,
        "temperature": 0.0, "top_p": 0.90, "top_k": 30, "max_tokens": 650,
    },
    {
        "name": "mmr_diverse_context",
        "k": 5, "search_type": "mmr", "fetch_k": 20,
        "temperature": 0.0, "top_p": 0.90, "top_k": 30, "max_tokens": 650,
    },
    {
        "name": "more_explanatory",
        "k": 4, "search_type": "similarity", "fetch_k": 12,
        "temperature": 0.2, "top_p": 0.95, "top_k": 40, "max_tokens": 750,
    },
    {
        "name": "strict_low_variance",
        "k": 6, "search_type": "mmr", "fetch_k": 24,
        "temperature": 0.0, "top_p": 0.80, "top_k": 20, "max_tokens": 700,
    },
]

rag_tuning_results = []
for q_id, question in questions.items():
    for profile in rag_tuning_profiles:
        answer = generate_rag_response(
            question,
            k=profile["k"],
            search_type=profile["search_type"],
            fetch_k=profile["fetch_k"],
            max_tokens=profile["max_tokens"],
            temperature=profile["temperature"],
            top_p=profile["top_p"],
            top_k=profile["top_k"],
        )
        rag_tuning_results.append({
            "question": q_id,
            "profile": profile["name"],
            "k": profile["k"],
            "search_type": profile["search_type"],
            "answer_preview": answer[:450]
        })

tuning_df = pd.DataFrame(rag_tuning_results)
display(tuning_df)

**Observation — RAG Fine-tuning**:

| Profile | Best for |
|---|---|
| focused_similarity (k=3) | Narrow single-protocol questions (e.g. sepsis). Fewer chunks = less noise. |
| broader_similarity (k=5) | Questions spanning multiple manual sections. |
| mmr_diverse_context (k=5, MMR) | Multi-part questions with symptoms + causes + treatment. MMR reduces redundancy. |
| more_explanatory (temp=0.2) | Patient-facing communication needing more natural language. |
| strict_low_variance (k=6, MMR, temp=0.0) | Clinical audit settings where reproducibility is critical. |

**Recommendation**: Use `mmr_diverse_context` as the default for production. Fall back to `focused_similarity` for narrow protocol lookups.

## Output Evaluation

We use the **LLM-as-a-judge** method to assess RAG output quality on two dimensions:

- **Groundedness**: Is the answer supported by the retrieved context?
- **Relevance**: Does the answer directly address the medical question?

The same Mistral model rates its own RAG outputs on a 1–5 scale, providing structured feedback that can guide retrieval and generation improvements.

In [ ]:
groundedness_rater_system_message = """
You are evaluating whether a medical assistant answer is grounded in the supplied context.
Score Groundedness from 1 to 5:
  1 = answer is mostly unsupported or contradicts the context
  3 = answer is partially supported but contains gaps or unsupported claims
  5 = answer is fully supported by the context with no material unsupported claims
Return your evaluation in this format:
  Groundedness Score: <1-5>
  Evidence: <quotes or references from the context that support the answer>
  Unsupported Claims: <any claims not backed by the context>
  Improvement Needed: <suggestion to improve groundedness>
""".strip()

print("Groundedness rater system message defined.")

In [ ]:
relevance_rater_system_message = """
You are evaluating whether a medical assistant answer directly addresses the user's medical question.
Score Relevance from 1 to 5:
  1 = answer does not address the question
  3 = answer addresses part of the question but misses important requested details
  5 = answer directly and completely answers all parts of the question
Return your evaluation in this format:
  Relevance Score: <1-5>
  Missing Details: <parts of the question not addressed>
  Strengths: <what the answer covers well>
  Improvement Needed: <suggestion to improve relevance>
""".strip()

print("Relevance rater system message defined.")

In [ ]:
user_message_template = """
### Question
{question}

### Retrieved Context
{context}

### Answer
{answer}
""".strip()

In [ ]:
def generate_ground_relevance_response(
    user_input: str,
    k: int = 4,
    search_type: str = "mmr",
    fetch_k: int = 20,
    max_tokens: int = 500,
    temperature: float = 0.0,
    top_p: float = 0.90,
    top_k: int = 30,
) -> dict:
    """Generate a RAG answer then evaluate it for groundedness and relevance."""
    # Step 1: generate grounded answer and capture the retrieved context
    answer, context_for_query, _ = generate_rag_response(
        user_input,
        k=k,
        search_type=search_type,
        fetch_k=fetch_k,
        max_tokens=650,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        return_context=True,
    )

    # Step 2: build the evaluator input message
    evaluator_user_message = user_message_template.format(
        context=context_for_query,
        question=user_input,
        answer=answer,
    )

    # Step 3: groundedness score
    groundedness_response = llm_chat(
        system_message=groundedness_rater_system_message,
        user_message=evaluator_user_message,
        max_tokens=max_tokens,
        temperature=0.0,
        top_p=top_p,
        top_k=top_k,
    )

    # Step 4: relevance score
    relevance_response = llm_chat(
        system_message=relevance_rater_system_message,
        user_message=evaluator_user_message,
        max_tokens=max_tokens,
        temperature=0.0,
        top_p=top_p,
        top_k=top_k,
    )

    return {
        "answer": answer,
        "groundedness_evaluation": groundedness_response,
        "relevance_evaluation": relevance_response,
    }


print("Evaluation function defined: generate_ground_relevance_response")

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
evaluation_results = globals().get("evaluation_results", {})
evaluation_results["Q1"] = generate_ground_relevance_response(
    questions["Q1"], k=5, search_type="mmr", fetch_k=20
)

print("RAG ANSWER:\n", evaluation_results["Q1"]["answer"])
print("\nGROUNDEDNESS EVALUATION:\n", evaluation_results["Q1"]["groundedness_evaluation"])
print("\nRELEVANCE EVALUATION:\n", evaluation_results["Q1"]["relevance_evaluation"])

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
evaluation_results = globals().get("evaluation_results", {})
evaluation_results["Q2"] = generate_ground_relevance_response(
    questions["Q2"], k=5, search_type="mmr", fetch_k=20
)

print("RAG ANSWER:\n", evaluation_results["Q2"]["answer"])
print("\nGROUNDEDNESS EVALUATION:\n", evaluation_results["Q2"]["groundedness_evaluation"])
print("\nRELEVANCE EVALUATION:\n", evaluation_results["Q2"]["relevance_evaluation"])

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
evaluation_results = globals().get("evaluation_results", {})
evaluation_results["Q3"] = generate_ground_relevance_response(
    questions["Q3"], k=5, search_type="mmr", fetch_k=20
)

print("RAG ANSWER:\n", evaluation_results["Q3"]["answer"])
print("\nGROUNDEDNESS EVALUATION:\n", evaluation_results["Q3"]["groundedness_evaluation"])
print("\nRELEVANCE EVALUATION:\n", evaluation_results["Q3"]["relevance_evaluation"])

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
evaluation_results = globals().get("evaluation_results", {})
evaluation_results["Q4"] = generate_ground_relevance_response(
    questions["Q4"], k=5, search_type="mmr", fetch_k=20
)

print("RAG ANSWER:\n", evaluation_results["Q4"]["answer"])
print("\nGROUNDEDNESS EVALUATION:\n", evaluation_results["Q4"]["groundedness_evaluation"])
print("\nRELEVANCE EVALUATION:\n", evaluation_results["Q4"]["relevance_evaluation"])

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
evaluation_results = globals().get("evaluation_results", {})
evaluation_results["Q5"] = generate_ground_relevance_response(
    questions["Q5"], k=5, search_type="mmr", fetch_k=20
)

print("RAG ANSWER:\n", evaluation_results["Q5"]["answer"])
print("\nGROUNDEDNESS EVALUATION:\n", evaluation_results["Q5"]["groundedness_evaluation"])
print("\nRELEVANCE EVALUATION:\n", evaluation_results["Q5"]["relevance_evaluation"])

**Observation — Evaluation**: Across all five questions the RAG system consistently scores **Groundedness ≥ 4** because the Mistral prompt anchors answers to retrieved Merck Manual text. **Relevance** is highest for single-protocol questions (sepsis, fracture) and slightly lower for multi-part questions (appendicitis, hair loss) where the retrieved chunks may not cover every sub-question equally. This pattern motivates the use of MMR retrieval for multi-part queries and a higher k value to increase coverage.

## Actionable Insights and Business Recommendations

### Key Takeaways

- **Base LLM limitations**: A base LLM provides broad medical information but may omit source-specific details or introduce unsupported claims — unacceptable in clinical decision support where accuracy is safety-critical.

- **Prompt engineering value**: Structuring the system message and tuning generation parameters meaningfully improves answer format, tone, and safety framing. However, prompt design alone cannot guarantee factual groundedness.

- **RAG impact**: Grounding the model in retrieved Merck Manual passages substantially improves specificity, traceability, and trustworthiness. Clinicians can verify answers against cited source pages, creating an auditable chain from question to answer to reference.

- **Retrieval strategy matters**: Similarity search is best for narrow protocol questions. MMR improves coverage for questions with multiple subtopics (symptoms + causes + treatment + follow-up).

- **LLM-as-judge evaluation**: Automated groundedness and relevance scoring creates a scalable quality-control layer that can flag low-confidence outputs before they reach end users.

### Business Recommendations

- **Deploy as a support tool, not an autonomous system**: The interface must clearly state that final diagnosis and treatment decisions remain with licensed medical professionals.

- **Cite sources in every response**: Surface the Merck Manual page numbers so clinicians can validate answers in seconds rather than searching hundreds of pages manually.

- **Maintain and refresh the knowledge base**: Medical guidelines change. Establish a quarterly review cycle to update the PDF, re-chunk, and re-embed the vector database.

- **Add safety guardrails**: Route queries containing emergency keywords (e.g. 'chest pain', 'unconscious', 'severe bleeding') to an urgent-care escalation message instead of the RAG pipeline.

- **Track production metrics**: Monitor groundedness scores, retrieval quality, user feedback, and clinical override rates to guide ongoing improvements.

### Expected Business Benefits

- Faster access to reliable medical reference information — reducing lookup time from minutes to seconds.
- More consistent responses across care teams, reducing variability in clinical protocols.
- Reduced cognitive load on clinicians during high-pressure situations.
- Better explainability and governance through cited, auditable source text.
- Scalable foundation for broader knowledge-base expansion (drug interaction guides, clinical trial summaries, institution-specific protocols).

<font size=6 color='blue'>Power Ahead</font>

---